# 06 — Major Backtest: Model Predictions vs Real Results

Parses `resultados_reais.md`, runs `predict_match()` for every match, and
compares the predicted winner against the actual result.

> **Note:** The model was trained on 2012–2020 HLTV data. Teams from recent
> majors may not exist in the lookup — those matches will use zeroed/imputed
> features and the prediction should be interpreted with caution.

Run `03_training.ipynb` first to generate the saved model.

In [ ]:
import re
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

from conf.settings import PROCESSED_DIR, DATASET_FILE
from src.module.model.model import CSWinnerModel
from src.module.model.features import build_features, build_team_stats_lookup

%matplotlib inline

## 1. Parse Match Results from Markdown

In [ ]:
MD_FILE = Path('resultados_reais.md')

def _parse_score(score: str):
    """Return (left, right) int tuple, or None if score contains '?'."""
    score = score.strip()
    if '?' in score:
        return None
    m = re.match(r'^(\d+)-(\d+)$', score)
    return (int(m.group(1)), int(m.group(2))) if m else None


def _winner_from_score(score: str):
    parsed = _parse_score(score)
    if parsed is None:
        return None
    l, r = parsed
    if l > r: return 'team1'
    if r > l: return 'team2'
    return None


def _fmt_from_score(score: str) -> str:
    """Infer bo1 vs bo3 from score magnitude."""
    parsed = _parse_score(score)
    if parsed and max(parsed) <= 2:
        return 'bo3'
    return 'bo1'


def parse_md_matches(path: Path) -> pd.DataFrame:
    """Parse all markdown tables in the file into a unified match DataFrame."""
    text = path.read_text(encoding='utf-8')
    lines = text.splitlines()

    matches = []
    current_section = 'Unknown'
    current_round = None
    current_bracket = None

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        # Track section headers (### ...)
        if line.startswith('#'):
            current_section = line.lstrip('#').strip()
            i += 1
            continue

        # Detect markdown table
        if line.startswith('|') and '|' in line[1:]:
            # Collect all rows of this table
            table_lines = []
            while i < len(lines) and lines[i].strip().startswith('|'):
                stripped = lines[i].strip()
                # Skip separator rows (|---|---|)
                if not re.match(r'^\|[-| :]+\|$', stripped):
                    table_lines.append(stripped)
                i += 1

            if len(table_lines) < 2:
                continue

            # Parse header
            header = [c.strip() for c in table_lines[0].split('|')[1:-1]]
            header_lower = [h.lower() for h in header]

            # Identify Score column
            score_idx = next(
                (j for j, h in enumerate(header_lower) if 'score' in h or 'map' in h.split()[-1:]),
                None
            )
            # Identify Team columns
            t1_idx = next((j for j, h in enumerate(header_lower) if 'team 1' in h or 'team1' in h), None)
            t2_idx = next((j for j, h in enumerate(header_lower) if 'team 2' in h or 'team2' in h), None)
            round_idx = next((j for j, h in enumerate(header_lower) if 'round' in h), None)
            bracket_idx = next((j for j, h in enumerate(header_lower) if 'bracket' in h), None)

            if t1_idx is None or t2_idx is None or score_idx is None:
                continue  # not a match table

            for row_line in table_lines[1:]:
                cells = [c.strip() for c in row_line.split('|')[1:-1]]
                if len(cells) != len(header):
                    continue

                team1 = cells[t1_idx]
                team2 = cells[t2_idx]
                score = cells[score_idx]
                rnd = cells[round_idx] if round_idx is not None else None
                bracket = cells[bracket_idx] if bracket_idx is not None else None

                winner = _winner_from_score(score)
                fmt = _fmt_from_score(score)
                # Treat explicit BO3 section as bo3
                if 'BO3' in current_section or 'BO3' in score:
                    fmt = 'bo3'

                if winner is None:
                    continue  # skip unknown/incomplete results

                matches.append({
                    'section': current_section,
                    'round': rnd,
                    'bracket': bracket,
                    'team1': team1,
                    'team2': team2,
                    'score': score,
                    'format': fmt,
                    'actual_winner': winner,
                    'actual_winner_name': team1 if winner == 'team1' else team2,
                })
        else:
            i += 1

    return pd.DataFrame(matches)


matches_df = parse_md_matches(MD_FILE)
print(f'Parsed {len(matches_df)} complete matches (skipped ones with "?").')
matches_df.head(10)

## 2. Load Model and Team Stats Lookup

In [ ]:
model = CSWinnerModel.load()
print('Model loaded.')

df_all = pd.read_parquet(PROCESSED_DIR / DATASET_FILE)
df_all = build_features(df_all)
lookup = build_team_stats_lookup(df_all)

print(f'Teams in lookup: {len(lookup)}')

## 3. Check Team Coverage

Teams not found in the lookup have no historical data — the model will use
imputed (mean) feature values for them, making predictions less reliable.

In [ ]:
all_teams = set(matches_df['team1'].unique()) | set(matches_df['team2'].unique())
in_lookup = {t for t in all_teams if t in lookup}
missing = all_teams - in_lookup

print(f'Teams in matches : {len(all_teams)}')
print(f'Found in lookup  : {len(in_lookup)}')
print(f'Missing (no data): {len(missing)}')
if missing:
    print('\nMissing teams (predictions use imputed features):')
    for t in sorted(missing):
        print(f'  - {t}')

## 4. Run Predictions for All Matches

In [ ]:
import logging
logging.disable(logging.WARNING)  # silence per-match model logs

results = []
for _, row in matches_df.iterrows():
    try:
        pred = model.predict_match(
            team1=row['team1'],
            team2=row['team2'],
            match_format=row['format'],
            stage=row['section'],
            team_stats=lookup,
        )
        predicted_side = 'team1' if pred['predicted_winner'] == row['team1'] else 'team2'
        results.append({
            **row.to_dict(),
            'predicted_winner': pred['predicted_winner'],
            'predicted_side': predicted_side,
            'prob_team1': pred['prob_team1_wins'],
            'prob_team2': pred['prob_team2_wins'],
            'elo_diff': pred['elo_diff'],
            'correct': predicted_side == row['actual_winner'],
            't1_in_lookup': row['team1'] in lookup,
            't2_in_lookup': row['team2'] in lookup,
        })
    except Exception as e:
        results.append({
            **row.to_dict(),
            'predicted_winner': 'ERROR',
            'predicted_side': None,
            'prob_team1': float('nan'),
            'prob_team2': float('nan'),
            'elo_diff': float('nan'),
            'correct': None,
            't1_in_lookup': row['team1'] in lookup,
            't2_in_lookup': row['team2'] in lookup,
        })

logging.disable(logging.NOTSET)

res = pd.DataFrame(results)
print(f'Predictions complete: {len(res)} matches.')

## 5. Full Results Table

In [ ]:
display_cols = ['section', 'round', 'bracket', 'team1', 'team2', 'score', 'format',
                'actual_winner_name', 'predicted_winner', 'correct',
                'prob_team1', 'prob_team2', 'elo_diff']

def _style_row(row):
    base = [''] * len(row)
    correct_idx = list(row.index).index('correct')
    if row['correct'] is True:
        base[correct_idx] = 'background-color: #d4edda; color: #155724'
    elif row['correct'] is False:
        base[correct_idx] = 'background-color: #f8d7da; color: #721c24'
    return base

(
    res[display_cols]
    .style
    .apply(_style_row, axis=1)
    .format({
        'prob_team1': '{:.1%}',
        'prob_team2': '{:.1%}',
        'elo_diff': '{:+.1f}',
    }, na_rep='-')
)

## 6. Overall Accuracy

In [ ]:
valid = res.dropna(subset=['correct'])
n_correct = valid['correct'].sum()
n_total = len(valid)
accuracy = n_correct / n_total if n_total > 0 else float('nan')

# Split by lookup coverage
both_known  = valid[valid['t1_in_lookup'] & valid['t2_in_lookup']]
any_unknown = valid[~(valid['t1_in_lookup'] & valid['t2_in_lookup'])]

print('=' * 45)
print(f'  Total matches evaluated : {n_total}')
print(f'  Correct predictions     : {int(n_correct)}')
print(f'  Overall accuracy        : {accuracy:.1%}')
print('=' * 45)
if len(both_known) > 0:
    acc_known = both_known['correct'].mean()
    print(f'  Both teams in lookup    : {len(both_known)} matches  →  {acc_known:.1%} accuracy')
if len(any_unknown) > 0:
    acc_unk = any_unknown['correct'].mean()
    print(f'  At least 1 team missing : {len(any_unknown)} matches  →  {acc_unk:.1%} accuracy')
print('=' * 45)

## 7. Accuracy by Stage / Section

In [ ]:
by_section = (
    valid
    .groupby('section')['correct']
    .agg(matches='count', correct='sum')
    .assign(accuracy=lambda d: d['correct'] / d['matches'])
    .sort_values('accuracy', ascending=False)
)
by_section.style.format({'accuracy': '{:.1%}'})

## 8. Confidence Analysis

How confident was the model when it was right vs wrong?

In [ ]:
# Confidence = max(prob_team1, prob_team2)
valid = res.dropna(subset=['correct', 'prob_team1', 'prob_team2']).copy()
valid['confidence'] = valid[['prob_team1', 'prob_team2']].max(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: confidence distribution — correct vs wrong
ax = axes[0]
for label, color, name in [(True, '#2ecc71', 'Correct'), (False, '#e74c3c', 'Wrong')]:
    subset = valid[valid['correct'] == label]['confidence']
    ax.hist(subset, bins=12, alpha=0.6, color=color, label=f'{name} ({len(subset)})', density=True)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Model confidence (max prob)')
ax.set_ylabel('Density')
ax.set_title('Confidence distribution: correct vs wrong')
ax.legend()

# Right: accuracy by confidence bucket
ax = axes[1]
buckets = pd.cut(valid['confidence'], bins=[0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 1.0],
                 labels=['40-50%', '50-55%', '55-60%', '60-65%', '65-70%', '70%+'])
bucket_acc = valid.groupby(buckets)['correct'].agg(['mean', 'count'])
bucket_acc['mean'].plot(kind='bar', ax=ax, color='steelblue', alpha=0.8)
ax.set_xlabel('Confidence bucket')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by confidence level')
ax.axhline(0.5, color='red', linestyle='--', linewidth=1)
ax.set_ylim(0, 1)
for p, (_, row_b) in zip(ax.patches, bucket_acc.iterrows()):
    ax.text(p.get_x() + p.get_width()/2, p.get_height() + 0.02,
            f'n={int(row_b["count"])}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

## 9. Biggest Upsets

Matches where the model was most confident but got it wrong.

In [ ]:
wrong = valid[valid['correct'] == False].copy()
wrong = wrong.sort_values('confidence', ascending=False)

print('Top 10 biggest upsets (model most confident, but wrong):')
(
    wrong[['section', 'team1', 'team2', 'score', 'actual_winner_name',
           'predicted_winner', 'prob_team1', 'prob_team2', 'confidence']]
    .head(10)
    .style
    .format({'prob_team1': '{:.1%}', 'prob_team2': '{:.1%}', 'confidence': '{:.1%}'})
    .background_gradient(subset=['confidence'], cmap='Reds')
)

## 10. Most Confident Correct Predictions

In [ ]:
correct = valid[valid['correct'] == True].copy()
correct = correct.sort_values('confidence', ascending=False)

print('Top 10 most confident correct predictions:')
(
    correct[['section', 'team1', 'team2', 'score', 'actual_winner_name',
             'predicted_winner', 'prob_team1', 'prob_team2', 'confidence']]
    .head(10)
    .style
    .format({'prob_team1': '{:.1%}', 'prob_team2': '{:.1%}', 'confidence': '{:.1%}'})
    .background_gradient(subset=['confidence'], cmap='Greens')
)

## 11. Per-Team Prediction Record

In [ ]:
# Build record: how often was each team's result correctly predicted?
team_rows = []
for _, row in valid.iterrows():
    for side, team in [('team1', row['team1']), ('team2', row['team2'])]:
        in_lkp = row[f't{side[-1]}_in_lookup'] if side == 'team1' else row['t2_in_lookup']
        team_rows.append({
            'team': team,
            'correct': row['correct'],
            'in_lookup': in_lkp,
        })

team_df = (
    pd.DataFrame(team_rows)
    .groupby(['team', 'in_lookup'])['correct']
    .agg(matches='count', correct='sum')
    .assign(accuracy=lambda d: d['correct'] / d['matches'])
    .sort_values('matches', ascending=False)
    .reset_index()
)

team_df.style.format({'accuracy': '{:.1%}'}).background_gradient(
    subset=['accuracy'], cmap='RdYlGn', vmin=0, vmax=1
)